In [1]:
import numpy as np
import pandas as pd
import json
import re
import csv
import ast

In [2]:
csop_data_csv = pd.read_csv('../raw_data/main_csop_data.csv')

In [3]:
# filter to only groups
csop_data_csv = csop_data_csv[csop_data_csv['group']==True]

In [4]:
csop_data_csv

,Unnamed: 0,game_id,round_id,round_index,task_index,complexity,group,type,score,optimal_solution,...,nominal_soln_pctile,real_soln_pctile,mean_nominal_pctile,mean_real_pctile,nominal_real_pctile_gap,mean_nominal_real_pctile_gap,num_violations_added,num_postcomplete_solns,zscore_num_postcomplete_solns,zscore_num_violations_added
20,20,C48peJSzquNCHsKSv,e7uDqY9mpTEZH5Q2b,1,4,High,True,HH,667,673,...,"[None, 1.0, 0.9146341463414634, 0.962962962962...","[None, 1.0, 0.9390243902439024, 0.271604938271...",0.770140,0.747532,[ 0. -0.02439024 0.69135802 0.6 ...,0.022608,11.0,13.0,NaN,NaN
21,21,C48peJSzquNCHsKSv,8m7bMTXhQH8pEEMfP,2,1,Very low,True,HH,340,340,...,"[None, 1.0, 0.5, 0.9523809523809523, 1.0, 1.0,...","[None, 1.0, 0.7727272727272727, 1.0, 0.7, 0.89...",0.764550,0.794511,[ 0. -0.27272727 -0.04761905 0.3 ...,-0.029961,4.0,7.0,NaN,NaN
22,22,C48peJSzquNCHsKSv,ogdaMhvwQDuX2jFnM,3,2,Low,True,HH,435,441,...,"[None, 0.717948717948718, 0.8947368421052632, ...","[None, 0.8205128205128205, 0.13157894736842105...",0.853681,0.638177,[-0.1025641 0.76315789 0.08108108 -0.027777...,0.215504,6.0,7.0,NaN,NaN
23,23,C48peJSzquNCHsKSv,oKguEirgCoNgfWsxf,4,3,Moderate,True,HH,672,672,...,"[None, 0.09433962264150944, 0.9811320754716981...","[None, 0.2830188679245283, 0.2830188679245283,...",0.772278,0.763269,[-0.18867925 0.69811321 -0.01923077 0.490196...,0.009009,11.0,22.0,NaN,NaN
24,24,C48peJSzquNCHsKSv,AC8SuSFZnSfst3Bvx,5,5,Very high,True,HH,867,996,...,"[None, 0.8951048951048951, 0.6830985915492958,...","[None, 0.916083916083916, 0.7323943661971831, ...",0.705725,0.790573,[-0.02097902 -0.04929577 -0.05673759 -0.121428...,-0.084847,10.0,20.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2145,2145,DhC75b28nAc9y5W9x,iMW6BXkH9jvcXnLgY,1,5,Very high,True,LL,785,996,...,"[None, 0.5034965034965035, 0.7887323943661971,...","[None, 0.5454545454545454, 0.8380281690140845,...",0.607252,0.754785,[-0.04195804 -0.04929577 -0.22695035 -0.078571...,-0.147533,9.0,3.0,NaN,NaN
2146,2146,DhC75b28nAc9y5W9x,HPxvosEMmRzqt526L,2,3,Moderate,True,LL,611,672,...,"[None, 0.9811320754716981, 0.05769230769230769...","[None, 0.2641509433962264, 0.5769230769230769,...",0.667604,0.765927,[ 0.71698113 -0.51923077 -0.01923077 -0.019607...,-0.098322,7.0,9.0,NaN,NaN
2147,2147,DhC75b28nAc9y5W9x,FEqJ3gzQYMK4hPsoz,3,2,Low,True,LL,441,441,...,"[None, 0.9743589743589743, 0.8947368421052632,...","[None, 0.9743589743589743, 0.9473684210526315,...",0.756416,0.877454,[ 0. -0.05263158 0. -0.055555...,-0.121037,4.0,14.0,NaN,NaN
2148,2148,DhC75b28nAc9y5W9x,9ZhXim8L3v2NbZQsy,4,1,Very low,True,LL,333,340,...,"[None, 0.782608695652174, 1.0, 1.0, 0.65, 1.0,...","[None, 0.782608695652174, 1.0, 0.5238095238095...",0.770094,0.718237,[ 0. 0. 0.47619048 -0.05 ...,0.051856,5.0,11.0,NaN,NaN


## List of Features to Include for CSOP

# Individual-Level Variations
- skill
- social_perceptivness
- cogStyleDiversity
- cogStyleSpeed

# Task-Related Features
- complexity

# Data Cleaning

In [5]:
# columns we want: game_id round_id, round_index, task_index, complexity, type, social_perceptiveness, skill, chat, normalized_score, zscore_score, zscore_round_duration, zscore_efficiency
csop_data_key_cols = csop_data_csv[["game_id", "round_id", "round_index", "task_index", "complexity", "type", "social_perceptiveness", "skill", "cogStyleDiversity", "cogStyleSpeed", "chat", "normalized_score", "score", "round_duration", "efficiency"]]

In [6]:
with open('../raw_data/csop_conversations_withblanks.csv', 'w', newline='') as csvfile:
    # start writing the header of the CSV
    fieldnames = ["batch_num", "round_num", "round_index", "task_index", "complexity", "type", "social_perceptiveness", "skill", "cogStyleDiversity", "cogStyleSpeed", "speaker_nickname", "message", "normalized_score", "score", "duration", "efficiency"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for index, row in csop_data_key_cols.iterrows():
        chat_list = ast.literal_eval(row['chat'])
        if(len(chat_list) > 0 ):
            # filter such that we only look at people who actually said something
            for chat in chat_list:
                writer.writerow({'batch_num': row['game_id'], 'round_num': row['round_id'], 'round_index': row['round_index'], 'task_index': row['task_index'], 'complexity': row['complexity'], 'type': row['type'], 'social_perceptiveness': row['social_perceptiveness'], 'skill': row['skill'], 'cogStyleDiversity': row['cogStyleDiversity'], 'cogStyleSpeed': row['cogStyleSpeed'], 'message': chat['text'], 'speaker_nickname': chat['playerId'], 'normalized_score': row['normalized_score'], 'score': row['score'], 'duration': row['round_duration'], 'efficiency': row['efficiency']})
        else: # no one said anything, so write a blank row
            BLANK_SPEAKER = "NULL_SPEAKER"
            writer.writerow({'batch_num': row['game_id'], 'round_num': row['round_id'], 'round_index': row['round_index'], 'task_index': row['task_index'], 'complexity': row['complexity'], 'type': row['type'], 'social_perceptiveness': row['social_perceptiveness'], 'skill': row['skill'], 'cogStyleDiversity': row['cogStyleDiversity'], 'cogStyleSpeed': row['cogStyleSpeed'], 'message': "", 'speaker_nickname': BLANK_SPEAKER, 'normalized_score': row['normalized_score'], 'score': row['score'], 'duration': row['round_duration'], 'efficiency': row['efficiency']})